# Multilingual Transformer Baselines for Arabic Deepfake-Text Detection

This experiment adds two multilingual baselines—**mBERT** and **XLM-RoBERTa**—to the existing AraBERTv2, MARBERTv2, and CAMeLBERT-DA comparison. Both platforms and all three preprocessing views use the same fixed split, seed, maximum length, effective batch size, epoch count, optimizer settings, and validation-based checkpoint rule as the Arabic-transformer experiments.

The notebook produces one graph-ready Excel workbook plus resumable per-run artifacts. It does not reproduce the broader ML, sequence-model, or publication-figure sections from the platform notebooks.


## Google Colab execution

Upload this notebook and the private `Dataset/` folder to the same Colab working directory. The expected layout is:

```text
/content/Arabic_Deepfake_Detection_Multilingual.ipynb
/content/Dataset/Twitter_data/Twitter_data/splits_rebuilt_v4/
/content/Dataset/Youtube_data/Youtube_data/splits_rebuilt/
```

Install the runtime packages once:

```python
%pip install -q "torch>=2.3" "transformers>=4.43,<5" pandas numpy scikit-learn emoji openpyxl
```

Training is enabled and requires a GPU runtime. Completed configurations are reused unless `OVERWRITE_COMPLETED_RUNS` is changed to `True`. The only final comparison artifact generated by this notebook is:

`Output/multilingual_transformers/multilingual_transformer_results.xlsx`


In [ ]:
%pip install -q "torch>=2.3" "transformers>=4.43,<5" pandas numpy scikit-learn emoji openpyxl

In [ ]:
from __future__ import annotations

import gc
import json
import math
import random
import re
import string
import time
import unicodedata
from copy import copy
from datetime import datetime, timezone
from pathlib import Path

import emoji
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
MODEL_SPECS = {
    "mbert": {
        "display_name": "mBERT",
        "model_id": "google-bert/bert-base-multilingual-cased",
    },
    "xlm_roberta": {
        "display_name": "XLM-R",
        "model_id": "FacebookAI/xlm-roberta-base",
    },
}
PLATFORMS = ("twitter", "youtube")
PREPROCESSING_ORDER = ("original", "manual", "deepfake_aware")
MAX_LEN = 128
MICRO_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
BSZ_VAL = BSZ_TEST = 64
N_EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 1e-3
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
CLASSIFIER_DROPOUT = 0.10
LABEL_SMOOTHING = 0.01
TOKENIZER_USE_FAST = True
BOOTSTRAP_REPEATS = 1000
RUN_TRAINING = True
OVERWRITE_COMPLETED_RUNS = False
STRICT_LEAKAGE_CHECK = True


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if RUN_TRAINING and DEVICE.type != "cuda":
    raise RuntimeError("RUN_TRAINING=True requires CUDA. Set it to False only for a CPU data/export dry run.")

PROJECT_ROOT = Path.cwd().resolve()
DATASET_ROOT = PROJECT_ROOT / "Dataset"
OUTPUT_ROOT = PROJECT_ROOT / "Output" / "multilingual_transformers"
RUNS_ROOT = OUTPUT_ROOT / "runs"
TABLE_ROOT = OUTPUT_ROOT / "tables"
for directory in (OUTPUT_ROOT, RUNS_ROOT, TABLE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_ROOTS = {
    "twitter": DATASET_ROOT / "Twitter_data" / "Twitter_data" / "splits_rebuilt_v4",
    "youtube": DATASET_ROOT / "Youtube_data" / "Youtube_data" / "splits_rebuilt",
}

print({
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "models": [spec["display_name"] for spec in MODEL_SPECS.values()],
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "output": str(OUTPUT_ROOT),
})


## Data and leakage checks

The private train/validation/test CSVs are loaded without rebuilding their partitions. Labels are normalized to `0 = Fake` and `1 = Real`. Exact test duplicates are removed from the corresponding training and validation split, then audited before training.


In [ ]:
LABEL_MAP = {"fake": 0, "deepfake": 0, "real": 1, "true": 1, "false": 0}


def normalize_whitespace_for_audit(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def resolve_dataset_paths(platform):
    base = SPLIT_ROOTS[platform]
    paths = {split: base / f"{split}_with_meta.csv" for split in ("train", "val", "test")}
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing private split files:\n" + "\n".join(missing))
    return paths


def read_split(path):
    frame = pd.read_csv(path, encoding="utf-8", on_bad_lines="skip")
    frame.columns = [str(column).strip() for column in frame.columns]
    text_col = "text" if "text" in frame else next(c for c in frame if c.lower() in {"text", "content", "body"})
    label_col = "label" if "label" in frame else next(c for c in frame if c.lower() in {"label", "type", "class"})
    frame = frame.rename(columns={text_col: "text", label_col: "label"}).copy()
    frame["text"] = frame["text"].map(normalize_whitespace_for_audit)
    raw_labels = frame["label"]
    mapped = raw_labels.astype(str).str.strip().str.lower().map(LABEL_MAP)
    numeric = pd.to_numeric(raw_labels, errors="coerce")
    frame["label"] = mapped.fillna(numeric)
    frame = frame.dropna(subset=["text", "label"])
    frame = frame[frame["text"].str.len() > 0].reset_index(drop=True)
    frame["label"] = frame["label"].astype(int)
    frame.insert(0, "sample_id", np.arange(len(frame), dtype=int))
    return frame


def load_and_deleak(platform):
    splits = {split: read_split(path) for split, path in resolve_dataset_paths(platform).items()}
    test_texts = set(splits["test"]["text"])
    removed = {}
    for split in ("train", "val"):
        before = len(splits[split])
        splits[split] = splits[split][~splits[split]["text"].isin(test_texts)].reset_index(drop=True)
        removed[split] = before - len(splits[split])
    return splits, removed


platform_splits = {}
split_rows, leakage_rows = [], []
for platform in PLATFORMS:
    platform_splits[platform], removed = load_and_deleak(platform)
    for split, frame in platform_splits[platform].items():
        split_rows.append({
            "platform": platform,
            "split": split,
            "rows": len(frame),
            "fake_rows": int((frame.label == 0).sum()),
            "real_rows": int((frame.label == 1).sum()),
            "removed_exact_test_duplicates": int(removed.get(split, 0)),
        })
    test_texts = set(platform_splits[platform]["test"]["text"])
    for split in ("train", "val"):
        overlap = len(set(platform_splits[platform][split]["text"]) & test_texts)
        leakage_rows.append({"platform": platform, "split": split, "test_overlap": overlap})

split_summary = pd.DataFrame(split_rows)
leakage_audit = pd.DataFrame(leakage_rows)
display(split_summary)
display(leakage_audit)
if RUN_TRAINING and STRICT_LEAKAGE_CHECK and leakage_audit["test_overlap"].sum() != 0:
    raise RuntimeError("Exact-text test leakage remains after de-duplication.")


## Source-faithful preprocessing

These compact functions preserve the same Twitter and YouTube rules used by the two platform notebooks: original text, strong manual cleaning, and deepfake-aware signal-preserving normalization. Odd-character patterns are learned from each platform's training split only.


In [ ]:
SPECIAL_TOKENS = {
    "url": "<URL>", "mention": "<MENTION>", "hashtag": "<HASHTAG>",
    "number": "<NUMBER>", "latin": "<LATIN>",
}
ALL_SPECIAL_TOKENS = list(SPECIAL_TOKENS.values())
SPECIAL_TOKEN_PATTERN = re.compile(r"(<URL>|<MENTION>|<HASHTAG>|<NUMBER>|<LATIN>)")
ARABIC_DIACRITICS_PATTERN = re.compile(r"[\u064B-\u065F\u0670]")
URL_PATTERN = re.compile(r"(?:https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")
HASHTAG_PATTERN = re.compile(r"(?<!\w)#[\w\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF_]+")
ENGLISH_PATTERN = re.compile(r"[A-Za-z]+(?:[A-Za-z0-9_./+-]*[A-Za-z0-9])?")
NUMBER_PATTERN = re.compile(r"(?<!\w)[\d\u0660-\u0669\u06F0-\u06F9]+(?:[.,:/\-][\d\u0660-\u0669\u06F0-\u06F9]+)*(?:[%٪])?")
HTML_PATTERN = re.compile(r"</?[^>\n]+>|&(?:[A-Za-z]+|#\d+|#x[0-9A-Fa-f]+);")
INVISIBLE_PATTERN = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]")
REPEATED_SEPARATOR_PATTERN = re.compile(r"([\-_=+~*#|/\\])\1+")
PUNCT_TRANSLATION = str.maketrans({",": "،", ";": "،", "؛": "،", "?": "؟", "!": ".", "…": "."})
ALEF_VARIANTS = {"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"}
YEH_VARIANTS = {"ى": "ي", "ی": "ي"}
HAMZA_VARIANTS = {"ؤ": "و", "ئ": "ي"}
MINIMAL_PUNCT = "،.؟"


def safe_str(text):
    return "" if pd.isna(text) else str(text)


def is_arabic_char(char):
    codepoint = ord(char)
    return any(lower <= codepoint <= upper for lower, upper in (
        (0x0600, 0x06FF), (0x0750, 0x077F), (0x08A0, 0x08FF),
        (0xFB50, 0xFDFF), (0xFE70, 0xFEFF),
    ))


def is_english_char(char):
    return "A" <= char <= "Z" or "a" <= char <= "z"


def apply_outside_tokens(text, transform):
    return "".join(part if SPECIAL_TOKEN_PATTERN.fullmatch(part or "") else transform(part)
                    for part in SPECIAL_TOKEN_PATTERN.split(safe_str(text)) if part)


def normalize_letters(text):
    for mapping in (ALEF_VARIANTS, YEH_VARIANTS, HAMZA_VARIANTS):
        for source, target in mapping.items():
            text = text.replace(source, target)
    return text


def remove_emojis(text):
    text = safe_str(text)
    if hasattr(emoji, "replace_emoji"):
        return emoji.replace_emoji(text, replace=" ")
    return emoji.get_emoji_regexp().sub(" ", text)


def normalize_repetition(text, max_repeat):
    return re.sub(r"(.)\1{" + str(max_repeat) + r",}", lambda match: match.group(1) * max_repeat, text)


def normalize_spacing(text):
    text = SPECIAL_TOKEN_PATTERN.sub(lambda match: f" {match.group(1)} ", text)
    text = re.sub(r"\s+([،.؟])", r"\1", text)
    text = re.sub(r"([،.؟])(?=\S)", r"\1 ", text)
    return re.sub(r"\s+", " ", text).strip()


def filter_chars(text, keep_tokens):
    parts = SPECIAL_TOKEN_PATTERN.split(text) if keep_tokens else [text]
    output = []
    for part in parts:
        if keep_tokens and SPECIAL_TOKEN_PATTERN.fullmatch(part or ""):
            output.append(part)
        else:
            output.append("".join(char for char in part if char.isspace() or is_arabic_char(char) or char.isdigit() or char in MINIMAL_PUNCT))
    return "".join(output)


def extract_odd_chars(texts):
    odd = set()
    for text in texts.fillna(""):
        for char in str(text):
            if char.isspace() or char in string.punctuation or is_arabic_char(char) or is_english_char(char) or char.isdigit():
                continue
            odd.add(char)
    return re.compile("[" + re.escape("".join(sorted(odd))) + "]") if odd else None


ODD_PATTERNS = {platform: extract_odd_chars(platform_splits[platform]["train"]["text"]) for platform in PLATFORMS}


def preprocess_text(text, platform, method):
    text = safe_str(text)
    if method == "original" or not text:
        return text
    text = unicodedata.normalize("NFKC", text)
    text = HTML_PATTERN.sub(" ", text)
    text = INVISIBLE_PATTERN.sub("", text)
    text = "".join(char for char in text if unicodedata.category(char) not in {"Cf", "Cc", "Cs"} or char in {"\n", "\t", "\r"})

    if method == "manual":
        text = URL_PATTERN.sub(" ", text)
        text = MENTION_PATTERN.sub(" ", text)
        text = HASHTAG_PATTERN.sub(" ", text)
        text = remove_emojis(text)
        text = ARABIC_DIACRITICS_PATTERN.sub("", text)
        text = normalize_letters(text).replace("ـ", "")
        text = normalize_repetition(text, 1)
        keep_tokens = False
    elif platform == "twitter":
        text = URL_PATTERN.sub(f" {SPECIAL_TOKENS['url']} ", text)
        text = MENTION_PATTERN.sub(f" {SPECIAL_TOKENS['mention']} ", text)
        text = HASHTAG_PATTERN.sub(f" {SPECIAL_TOKENS['hashtag']} ", text)
        text = apply_outside_tokens(text, lambda part: NUMBER_PATTERN.sub(f" {SPECIAL_TOKENS['number']} ", part))
        text = apply_outside_tokens(text, lambda part: ENGLISH_PATTERN.sub(f" {SPECIAL_TOKENS['latin']} ", part))
        text = remove_emojis(text)
        text = normalize_letters(text).replace("ـ", "")
        text = apply_outside_tokens(text, lambda part: normalize_repetition(part, 2))
        keep_tokens = True
    else:
        text = URL_PATTERN.sub(" ", text)
        text = MENTION_PATTERN.sub(" ", text)
        text = HASHTAG_PATTERN.sub(" ", text)
        text = remove_emojis(text)
        text = apply_outside_tokens(text, lambda part: NUMBER_PATTERN.sub(f" {SPECIAL_TOKENS['number']} ", part))
        text = apply_outside_tokens(text, lambda part: ENGLISH_PATTERN.sub(" ", part))
        text = ARABIC_DIACRITICS_PATTERN.sub("", text).replace("ـ", "")
        text = apply_outside_tokens(text, lambda part: normalize_repetition(part, 2))
        keep_tokens = True

    text = text.translate(PUNCT_TRANSLATION)
    text = re.sub(r"،{2,}", "،", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"؟{2,}", "؟", text)
    text = apply_outside_tokens(text, lambda part: REPEATED_SEPARATOR_PATTERN.sub(" ", part))
    if ODD_PATTERNS[platform] is not None:
        text = ODD_PATTERNS[platform].sub(" ", text)
    if method == "manual":
        text = apply_outside_tokens(text, lambda part: ENGLISH_PATTERN.sub(" ", part))
    return normalize_spacing(filter_chars(text, keep_tokens=keep_tokens))


def required_special_tokens(platform, preprocessing):
    if preprocessing != "deepfake_aware":
        return []
    return ALL_SPECIAL_TOKENS if platform == "twitter" else [SPECIAL_TOKENS["number"]]


In [ ]:
prepared_splits = {}
preprocessing_rows = []
for platform in PLATFORMS:
    prepared_splits[platform] = {}
    for split, raw_frame in platform_splits[platform].items():
        frame = raw_frame.copy()
        for preprocessing in PREPROCESSING_ORDER:
            column = f"text_{preprocessing}"
            frame[column] = frame["text"].map(lambda value, p=platform, m=preprocessing: preprocess_text(value, p, m))
            preprocessing_rows.append({
                "platform": platform,
                "split": split,
                "preprocessing": preprocessing,
                "rows": len(frame),
                "empty_rows": int((frame[column].str.len() == 0).sum()),
                "changed_pct": 0.0 if preprocessing == "original" else 100 * float((frame[column] != frame["text_original"] if "text_original" in frame else frame[column] != frame["text"]).mean()),
                "mean_tokens": float(frame[column].str.split().map(len).mean()),
            })
        prepared_splits[platform][split] = frame

preprocessing_audit = pd.DataFrame(preprocessing_rows)
display(preprocessing_audit)


## Fine-tuning and evaluation

Each model is trained separately for every platform/preprocessing combination. Gradient accumulation preserves an effective batch size of 64 on a 16 GB GPU. The validation-best checkpoint is selected by Macro-F1, with validation loss and then earlier epoch as deterministic tie-breakers. Test data are evaluated once after selection.


In [ ]:
class TextDataset(Dataset):
    def __init__(self, frame, text_column):
        self.texts = frame[text_column].fillna("").astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.indices = np.arange(len(frame), dtype=int)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.texts[index], self.labels[index], int(self.indices[index])


def make_collator(tokenizer):
    def collate(records):
        texts, labels, indices = zip(*records)
        encoded = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors="pt",
        )
        encoded["labels"] = torch.tensor(labels, dtype=torch.long)
        encoded["sample_idx"] = torch.tensor(indices, dtype=torch.long)
        return encoded
    return collate


def make_loader(frame, text_column, tokenizer, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TextDataset(frame, text_column), batch_size=batch_size, shuffle=shuffle,
        collate_fn=make_collator(tokenizer), generator=generator if shuffle else None,
        num_workers=0, pin_memory=torch.cuda.is_available(),
    )


def move_batch(batch):
    labels = batch.pop("labels").to(DEVICE, non_blocking=True)
    indices = batch.pop("sample_idx").cpu().numpy()
    inputs = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
    return inputs, labels, indices


def classification_metrics(y_true, y_pred):
    macro = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro[0]),
        "macro_recall": float(macro[1]),
        "macro_f1": float(macro[2]),
        "weighted_precision": float(weighted[0]),
        "weighted_recall": float(weighted[1]),
        "weighted_f1": float(weighted[2]),
    }


def bootstrap_macro_f1(y_true, y_pred, seed):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(BOOTSTRAP_REPEATS):
        indices = rng.integers(0, len(y_true), len(y_true))
        values.append(precision_recall_fscore_support(
            y_true[indices], y_pred[indices], average="macro", zero_division=0,
        )[2])
    return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_rows = 0.0, 0
    logits_all, labels_all, indices_all = [], [], []
    for batch in loader:
        inputs, labels, indices = move_batch(batch)
        logits = model(**inputs).logits
        total_loss += float(F.cross_entropy(logits, labels, reduction="sum"))
        total_rows += len(labels)
        logits_all.append(logits.cpu())
        labels_all.append(labels.cpu())
        indices_all.append(indices)
    logits = torch.cat(logits_all).numpy()
    labels = torch.cat(labels_all).numpy()
    indices = np.concatenate(indices_all)
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predictions = probabilities.argmax(axis=1)
    return {
        "loss": total_loss / max(1, total_rows),
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities,
        "indices": indices,
        **classification_metrics(labels, predictions),
    }


def build_model_and_tokenizer(model_id, platform, preprocessing):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=TOKENIZER_USE_FAST)
    config = AutoConfig.from_pretrained(model_id, num_labels=2)
    for attribute in ("hidden_dropout_prob", "attention_probs_dropout_prob", "classifier_dropout"):
        if hasattr(config, attribute):
            setattr(config, attribute, CLASSIFIER_DROPOUT)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, config=config)
    new_tokens = [token for token in required_special_tokens(platform, preprocessing) if token not in tokenizer.get_vocab()]
    if new_tokens:
        tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
        model.resize_token_embeddings(len(tokenizer))
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    return model.to(DEVICE), tokenizer, new_tokens


def run_paths(platform, model_key, preprocessing):
    run_id = f"{platform}__{model_key}__{preprocessing}__maxlen={MAX_LEN}__seed={SEED}"
    run_dir = RUNS_ROOT / run_id
    return run_dir, {
        "metrics": run_dir / "overall_metrics.json",
        "per_class": run_dir / "per_class_metrics.csv",
        "confusion": run_dir / "confusion_matrix.csv",
        "predictions": run_dir / "test_predictions.csv",
        "history": run_dir / "training_history.csv",
        "metadata": run_dir / "run_metadata.json",
        "complete": run_dir / "COMPLETED.json",
    }


def load_completed_run(paths):
    if not paths["complete"].exists():
        return None
    return {
        "metrics": pd.DataFrame([json.loads(paths["metrics"].read_text(encoding="utf-8"))]),
        "per_class": pd.read_csv(paths["per_class"]),
        "confusion": pd.read_csv(paths["confusion"]),
        "predictions": pd.read_csv(paths["predictions"]),
        "history": pd.read_csv(paths["history"]),
        "manifest": pd.DataFrame([json.loads(paths["metadata"].read_text(encoding="utf-8"))]),
    }


def train_configuration(platform, model_key, preprocessing, run_number):
    spec = MODEL_SPECS[model_key]
    run_dir, paths = run_paths(platform, model_key, preprocessing)
    if not OVERWRITE_COMPLETED_RUNS:
        completed = load_completed_run(paths)
        if completed is not None:
            print(f"[reuse] {run_dir.name}")
            return completed

    seed_everything(SEED)
    run_dir.mkdir(parents=True, exist_ok=True)
    text_column = f"text_{preprocessing}"
    model, tokenizer, added_tokens = build_model_and_tokenizer(spec["model_id"], platform, preprocessing)
    train_loader = make_loader(prepared_splits[platform]["train"], text_column, tokenizer, MICRO_BATCH_SIZE, True)
    val_loader = make_loader(prepared_splits[platform]["val"], text_column, tokenizer, BSZ_VAL, False)
    test_loader = make_loader(prepared_splits[platform]["test"], text_column, tokenizer, BSZ_TEST, False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    total_steps = steps_per_epoch * N_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, max(1, int(WARMUP_RATIO * total_steps)), total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best = {"macro_f1": -1.0, "loss": float("inf"), "epoch": 0}
    history_rows = []
    training_start = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_loss = 0.0
        epoch_start = time.time()
        for batch_number, batch in enumerate(train_loader, start=1):
            inputs, labels, _ = move_batch(batch)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(**inputs).logits
                loss = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTHING)
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            if batch_number % GRADIENT_ACCUMULATION_STEPS == 0 or batch_number == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            train_loss += float(loss.detach())

        validation = evaluate(model, val_loader)
        history_rows.append({
            "platform": platform, "model_name": spec["display_name"], "model_key": model_key,
            "preprocessing": preprocessing, "epoch": epoch,
            "train_loss": train_loss / max(1, len(train_loader)),
            "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
            "val_macro_f1": validation["macro_f1"], "epoch_time_sec": time.time() - epoch_start,
        })
        better = validation["macro_f1"] > best["macro_f1"] + 1e-12
        tied = abs(validation["macro_f1"] - best["macro_f1"]) <= 1e-12 and validation["loss"] < best["loss"] - 1e-12
        if better or tied:
            best = {"macro_f1": validation["macro_f1"], "loss": validation["loss"], "epoch": epoch}
            model.save_pretrained(run_dir / "best_model")
            tokenizer.save_pretrained(run_dir / "best_model")
        pd.DataFrame(history_rows).to_csv(paths["history"], index=False)
        print(f"{run_dir.name} | epoch {epoch}/{N_EPOCHS} | val Macro-F1={validation['macro_f1']:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    model = AutoModelForSequenceClassification.from_pretrained(run_dir / "best_model").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(run_dir / "best_model", use_fast=TOKENIZER_USE_FAST)
    test_loader = make_loader(prepared_splits[platform]["test"], text_column, tokenizer, BSZ_TEST, False)
    test = evaluate(model, test_loader)
    ci_low, ci_high = bootstrap_macro_f1(test["labels"], test["predictions"], SEED + run_number)
    total_time = time.time() - training_start

    metric_row = {
        "platform": platform, "evaluation_scope": "in_domain",
        "source_platform": platform, "target_platform": platform,
        "model_family": "Transformer", "model_scope": "Multilingual",
        "model_name": spec["display_name"], "model_key": model_key,
        "model_id": spec["model_id"], "preprocessing": preprocessing,
        "max_len": MAX_LEN, "seed": SEED, "best_epoch": best["epoch"],
        **{key: test[key] for key in (
            "accuracy", "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
        )},
        "macro_f1_ci_low": ci_low, "macro_f1_ci_high": ci_high,
        "test_loss": test["loss"], "training_time_sec": total_time,
        "status": "observed",
    }

    class_metrics = precision_recall_fscore_support(
        test["labels"], test["predictions"], labels=[0, 1], zero_division=0,
    )
    per_class = pd.DataFrame({
        "class_id": [0, 1], "class_name": ["Fake", "Real"],
        "precision": class_metrics[0], "recall": class_metrics[1],
        "f1": class_metrics[2], "support": class_metrics[3].astype(int),
    })
    for key, value in {
        "platform": platform, "model_name": spec["display_name"], "model_key": model_key,
        "preprocessing": preprocessing,
    }.items():
        per_class.insert(0, key, value)

    matrix = confusion_matrix(test["labels"], test["predictions"], labels=[0, 1])
    confusion = pd.DataFrame([{
        "platform": platform, "model_name": spec["display_name"], "model_key": model_key,
        "preprocessing": preprocessing,
        "true_fake_pred_fake": int(matrix[0, 0]), "true_fake_pred_real": int(matrix[0, 1]),
        "true_real_pred_fake": int(matrix[1, 0]), "true_real_pred_real": int(matrix[1, 1]),
    }])

    predictions = pd.DataFrame({
        "sample_idx": test["indices"], "y_true": test["labels"], "y_pred": test["predictions"],
        "prob_fake": test["probabilities"][:, 0], "prob_real": test["probabilities"][:, 1],
    })
    predictions["correct"] = predictions["y_true"] == predictions["y_pred"]
    predictions["error_type"] = np.select(
        [(predictions.y_true == 0) & (predictions.y_pred == 1), (predictions.y_true == 1) & (predictions.y_pred == 0)],
        ["false_real", "false_fake"], default="correct",
    )
    predictions["predicted_confidence"] = np.where(
        predictions["y_pred"] == 1, predictions["prob_real"], predictions["prob_fake"]
    )
    metadata_columns = [column for column in (
        "sample_id", "deception_type", "deception_category", "category", "source", "source_id", "parent_id", "pair_id",
    ) if column in prepared_splits[platform]["test"].columns]
    source_rows = prepared_splits[platform]["test"].iloc[test["indices"]].reset_index(drop=True)
    for column in metadata_columns:
        predictions[column] = source_rows[column].to_numpy()
    predictions["text"] = source_rows["text"].to_numpy()
    predictions["processed_text"] = source_rows[text_column].to_numpy()
    for key, value in {
        "platform": platform, "model_name": spec["display_name"], "model_key": model_key,
        "preprocessing": preprocessing,
    }.items():
        predictions.insert(0, key, value)

    manifest = {
        "platform": platform, "model_name": spec["display_name"], "model_key": model_key,
        "model_id": spec["model_id"], "preprocessing": preprocessing,
        "seed": SEED, "max_len": MAX_LEN, "micro_batch_size": MICRO_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE, "epochs": N_EPOCHS,
        "learning_rate": LR, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
        "classifier_dropout": CLASSIFIER_DROPOUT, "label_smoothing": LABEL_SMOOTHING,
        "best_epoch": best["epoch"], "added_special_tokens": added_tokens,
        "selection_rule": "max validation Macro-F1; tie lower validation loss; then earlier epoch",
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    paths["metrics"].write_text(json.dumps(metric_row, indent=2), encoding="utf-8")
    per_class.to_csv(paths["per_class"], index=False)
    confusion.to_csv(paths["confusion"], index=False)
    predictions.to_csv(paths["predictions"], index=False, encoding="utf-8")
    pd.DataFrame(history_rows).to_csv(paths["history"], index=False)
    paths["metadata"].write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["complete"].write_text(json.dumps({"completed_utc": manifest["completed_utc"]}, indent=2), encoding="utf-8")
    return {
        "metrics": pd.DataFrame([metric_row]), "per_class": per_class,
        "confusion": confusion, "predictions": predictions,
        "history": pd.DataFrame(history_rows), "manifest": pd.DataFrame([manifest]),
    }


## Run the twelve controlled configurations

The loop is intentionally simple and sequential to avoid GPU-memory contention. Restarting the notebook reuses every run that has a completion marker.


In [ ]:
artifact_groups = {key: [] for key in ("metrics", "per_class", "confusion", "predictions", "history", "manifest")}

if RUN_TRAINING:
    configurations = [
        (platform, model_key, preprocessing)
        for platform in PLATFORMS
        for model_key in MODEL_SPECS
        for preprocessing in PREPROCESSING_ORDER
    ]
    for run_number, (platform, model_key, preprocessing) in enumerate(configurations, start=1):
        print(f"\n[{run_number}/{len(configurations)}] {platform} | {model_key} | {preprocessing}")
        artifacts = train_configuration(platform, model_key, preprocessing, run_number)
        for key in artifact_groups:
            artifact_groups[key].append(artifacts[key])

overall_metrics = pd.concat(artifact_groups["metrics"], ignore_index=True) if artifact_groups["metrics"] else pd.DataFrame()
per_class_metrics = pd.concat(artifact_groups["per_class"], ignore_index=True) if artifact_groups["per_class"] else pd.DataFrame()
confusion_matrices = pd.concat(artifact_groups["confusion"], ignore_index=True) if artifact_groups["confusion"] else pd.DataFrame()
all_predictions = pd.concat(artifact_groups["predictions"], ignore_index=True) if artifact_groups["predictions"] else pd.DataFrame()
training_history = pd.concat(artifact_groups["history"], ignore_index=True) if artifact_groups["history"] else pd.DataFrame()
run_manifest = pd.concat(artifact_groups["manifest"], ignore_index=True) if artifact_groups["manifest"] else pd.DataFrame()

if overall_metrics.empty:
    display(Markdown("**No observed results yet.** Enable training on CUDA to populate the workbook."))
else:
    display(overall_metrics.sort_values(["platform", "model_name", "preprocessing"]))


## Misclassification outputs

Only incorrect test predictions are exported with their raw and processed text. Summary tables retain error direction and available deception metadata without adding extra plots.


In [ ]:
if all_predictions.empty:
    misclassifications = pd.DataFrame()
    misclassification_summary = pd.DataFrame()
else:
    misclassifications = all_predictions[~all_predictions["correct"]].copy()
    grouping = ["platform", "model_name", "model_key", "preprocessing", "error_type"]
    deception_column = next((column for column in ("deception_type", "deception_category", "category") if column in misclassifications), None)
    if deception_column:
        grouping.append(deception_column)
    misclassification_summary = (
        misclassifications.groupby(grouping, dropna=False, as_index=False)
        .agg(error_count=("y_true", "size"), mean_error_confidence=("predicted_confidence", "mean"))
    )

display(misclassification_summary)


## Excel export

This workbook contains only the newly observed mBERT and XLM-R results. The `plot_data` sheet is the canonical long-format input that a separate plotting notebook can later merge with the existing AraBERTv2, MARBERTv2, and CAMeLBERT-DA results.

After the Colab run finishes, download the workbook from the Files panel or execute:

```python
from google.colab import files
files.download(str(workbook_path))
```


In [ ]:
MODEL_ORDER = {"mBERT": 1, "XLM-R": 2}
PREPROCESSING_SEQUENCE = {"original": 1, "manual": 2, "deepfake_aware": 3}

plot_columns = [
    "platform", "model_name", "model_key", "model_id", "model_scope",
    "preprocessing", "accuracy", "macro_precision", "macro_recall",
    "macro_f1", "weighted_f1", "macro_f1_ci_low", "macro_f1_ci_high",
    "seed", "best_epoch", "training_time_sec", "status",
]
plot_data = overall_metrics.reindex(columns=plot_columns).copy()
if not plot_data.empty:
    plot_data["model_order"] = plot_data["model_name"].map(MODEL_ORDER)
    plot_data["preprocessing_order"] = plot_data["preprocessing"].map(PREPROCESSING_SEQUENCE)
    plot_data = plot_data.sort_values(
        ["platform", "model_order", "preprocessing_order"]
    ).reset_index(drop=True)

results_status = pd.DataFrame([{
    "expected_runs": len(PLATFORMS) * len(MODEL_SPECS) * len(PREPROCESSING_ORDER),
    "completed_runs": len(overall_metrics),
    "is_complete": len(overall_metrics) == len(PLATFORMS) * len(MODEL_SPECS) * len(PREPROCESSING_ORDER),
    "models": ", ".join(spec["display_name"] for spec in MODEL_SPECS.values()),
    "platforms": ", ".join(PLATFORMS),
    "preprocessing_views": ", ".join(PREPROCESSING_ORDER),
}])

protocol = pd.DataFrame([
    {"field": "Purpose", "value": "Generate standalone mBERT and XLM-R results for later five-transformer plotting"},
    {"field": "Platforms", "value": ", ".join(PLATFORMS)},
    {"field": "Preprocessing", "value": ", ".join(PREPROCESSING_ORDER)},
    {"field": "Primary metric", "value": "Test Macro-F1 after validation-based checkpoint selection"},
    {"field": "Models", "value": ", ".join(spec["display_name"] for spec in MODEL_SPECS.values())},
    {"field": "Seed", "value": SEED},
    {"field": "Maximum sequence length", "value": MAX_LEN},
    {"field": "Micro / effective batch", "value": f"{MICRO_BATCH_SIZE} / {EFFECTIVE_BATCH_SIZE}"},
    {"field": "Epochs", "value": N_EPOCHS},
    {"field": "Learning rate", "value": LR},
    {"field": "Selection rule", "value": "Maximum validation Macro-F1; tie lower validation loss; then earlier epoch"},
    {"field": "Result scope", "value": "Multilingual transformers only; no existing Arabic-transformer files are required"},
])

tables = {
    "README": protocol,
    "results_status": results_status,
    "split_summary": split_summary,
    "leakage_audit": leakage_audit,
    "preprocessing_audit": preprocessing_audit,
    "overall_metrics": overall_metrics,
    "plot_data": plot_data,
    "per_class_metrics": per_class_metrics,
    "confusion_matrices": confusion_matrices,
    "misclass_summary": misclassification_summary,
    "misclassifications": misclassifications,
    "training_history": training_history,
    "run_manifest": run_manifest,
}

workbook_path = OUTPUT_ROOT / "multilingual_transformer_results.xlsx"
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    for sheet_name, table in tables.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        worksheet = writer.sheets[sheet_name[:31]]
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.sheet_view.showGridLines = False
        for cell in worksheet[1]:
            header_font = copy(cell.font)
            header_font.bold = True
            header_font.color = "FFFFFF"
            cell.font = header_font
            header_fill = copy(cell.fill)
            header_fill.fill_type = "solid"
            header_fill.fgColor.rgb = "1F4E78"
            cell.fill = header_fill
        for column_cells in worksheet.columns:
            values = [str(cell.value) if cell.value is not None else "" for cell in column_cells[:200]]
            worksheet.column_dimensions[column_cells[0].column_letter].width = min(
                max(11, max(map(len, values), default=0) + 2), 45
            )

print(f"Saved workbook: {workbook_path}")
print("Colab download command: files.download(str(workbook_path))")
display(results_status)
display(plot_data)


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/Arabic_deepfake_detection"
)

OUTPUT_ROOT = DRIVE_PROJECT / "Output" / "multilingual_transformers"
RUNS_ROOT = OUTPUT_ROOT / "runs"
TABLES_ROOT = OUTPUT_ROOT / "tables"

for folder in (OUTPUT_ROOT, RUNS_ROOT, TABLES_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_ROOT}")

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive", force_remount=True)

local_output = Path("/content/Output/multilingual_transformers")
drive_output = Path(
    "/content/drive/MyDrive/Arabic_deepfake_detection/"
    "Output/multilingual_transformers"
)

if not local_output.exists():
    raise FileNotFoundError(f"Local output not found: {local_output}")

drive_output.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    local_output,
    drive_output,
    dirs_exist_ok=True,
)

# Verify copied files using relative paths and file sizes
local_files = {
    file.relative_to(local_output): file.stat().st_size
    for file in local_output.rglob("*")
    if file.is_file()
}

drive_files = {
    file.relative_to(drive_output): file.stat().st_size
    for file in drive_output.rglob("*")
    if file.is_file()
}

missing_files = sorted(set(local_files) - set(drive_files))
size_mismatches = sorted(
    path for path in local_files
    if path in drive_files and local_files[path] != drive_files[path]
)

print(f"Local files: {len(local_files)}")
print(f"Google Drive files: {len(drive_files)}")
print(f"Missing files: {len(missing_files)}")
print(f"Size mismatches: {len(size_mismatches)}")

if missing_files:
    print("Missing:", missing_files)

if size_mismatches:
    print("Size mismatches:", size_mismatches)

assert not missing_files, "Some files were not copied."
assert not size_mismatches, "Some copied files are incomplete."

print(f"\nSuccessfully copied and verified:\n{drive_output}")